In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

import time
import sys
import requests
import logging
import os

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta
from scipy import stats
from scipy.optimize import minimize
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from statsmodels.tsa.api import SimpleExpSmoothing, Holt, ExponentialSmoothing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_excel("pmovdcE 23 4May26.xlsx", skiprows=4)
df["P/N"] = df["P/N"].astype(str)

# -----------------------------
# CONFIG
# -----------------------------
pn_2867 = "CC 2867"
pn_2868 = "CC 2868"
pn_2869 = "CC 2869"

d_cols = [f"D-{i}" for i in range(1, 17)]
call_cols = [c for c in df.columns if c.startswith("C-") or c == "C TM"]

# -----------------------------
# HELPER
# -----------------------------
def get_row(brc, pn):
    rows = df[(df["Brc"] == brc) & (df["P/N"] == pn)]
    if not rows.empty:
        return rows.iloc[0]
    return pd.Series(0, index=df.columns)

# -----------------------------
# BUILD CC 2867A
# -----------------------------
new_rows = []

for brc in df["Brc"].dropna().unique():

    r2867 = get_row(brc, pn_2867)
    r2868 = get_row(brc, pn_2868)
    r2869 = get_row(brc, pn_2869)

    # Start with all blank
    new_row = pd.Series(index=df.columns, dtype="object")

    new_row["Brc"] = brc
    new_row["P/N"] = "CC 2867A"
    new_row["Desc"] = "Total of CC2867"
    new_row["D TM"] = 0

    # DN Price
    new_row["DN Price"] = 3038.49

    new_row["OH"] = round(
        (((r2869["OH"] * 20) + (r2868["OH"] * 200)) / 1000) + r2867["OH"], 0
    )

    new_row["OO"] = round(
        (((r2869["OO"] * 20) + (r2868["OO"] * 200)) / 1000) + r2867["OO"], 0
    )

    for col in d_cols:
        new_row[col] = round(
            (((r2869[col] * 20) + (r2868[col] * 200)) / 1000) + r2867[col],
            0
        )

    #Agc
    new_row["Agc"] = 23
    # Book / Alloc
    new_row["Book"] = 0
    new_row["Alloc\nIn"] = 0
    new_row["Alloc\nOut"] = 0

    # Call columns
    for col in call_cols:
        new_row[col] = round(
            (((r2869[col] * 20) + (r2868[col] * 200)) / 1000) + r2867[col],
            0
        )

    new_rows.append(new_row)

# -----------------------------
# APPEND
# -----------------------------
df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)


C:\Users\Brandon\AppData\Local\Temp\ipykernel_37852\3492041129.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)
C:\Users\Brandon\AppData\Local\Temp\ipykernel_37852\3492041129.py:81: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(new_rows)], ignore_index=True)


In [4]:
# =========================================================
# EXPORT
# =========================================================
df.to_excel("pmovdcE 23 4May26 CC2867A.xlsx", index=False)